In [2]:
import cv2

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()

    cv2.imshow("Camera", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [5]:
import cv2
import mediapipe as mp

# MediaPipe classes
BaseOptions = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

# Settings for Hand Landmarker
options = HandLandmarkerOptions(
    base_options=BaseOptions(
        model_asset_path="hand_landmarker.task"
    ),
    running_mode=VisionRunningMode.VIDEO,
    num_hands=1,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5
)

# Create hand detector
with HandLandmarker.create_from_options(options) as landmarker:

    # Open computer camera
    cap = cv2.VideoCapture(0)

    frame_timestamp = 0

    while True:

        # Read camera frame
        ret, frame = cap.read()

        if not ret:
            break

        # Convert BGR → RGB
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Convert OpenCV image to MediaPipe Image
        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb_frame
        )

        # Detect hand
        result = landmarker.detect_for_video(
            mp_image,
            frame_timestamp
        )

        frame_timestamp += 1

        # Check whether a hand was detected
        if result.hand_landmarks:

            # Only one hand because num_hands=1
            landmarks = result.hand_landmarks[0]

            # Draw all 21 landmarks
            for i, landmark in enumerate(landmarks):

                # Convert normalized coordinates to pixels
                x = int(landmark.x * frame.shape[1])
                y = int(landmark.y * frame.shape[0])

                # Draw point
                cv2.circle(
                    frame,
                    (x, y),
                    5,
                    (0, 255, 0),
                    -1
                )

                # Display landmark number
                cv2.putText(
                    frame,
                    str(i),
                    (x + 5, y - 5),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    (255, 0, 0),
                    1
                )

            # Draw connections between landmarks
            connections = [
                (0,1), (1,2), (2,3), (3,4),
                (0,5), (5,6), (6,7), (7,8),
                (5,9), (9,10), (10,11), (11,12),
                (9,13), (13,14), (14,15), (15,16),
                (13,17), (17,18), (18,19), (19,20),
                (0,17)
            ]

            for start, end in connections:

                x1 = int(landmarks[start].x * frame.shape[1])
                y1 = int(landmarks[start].y * frame.shape[0])

                x2 = int(landmarks[end].x * frame.shape[1])
                y2 = int(landmarks[end].y * frame.shape[0])

                cv2.line(
                    frame,
                    (x1, y1),
                    (x2, y2),
                    (0, 255, 0),
                    2
                )

        # Show camera
        cv2.imshow("MediaPipe Hand Landmarks", frame)

        # Press q to quit
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

In [3]:
import cv2
import mediapipe as mp
import numpy as np
import os
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report


# =========================================================
# 1. MEDIA PIPE SETUP
# =========================================================

BaseOptions = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
RunningMode = mp.tasks.vision.RunningMode


# =========================================================
# 2. FUNCTION TO CONVERT ONE IMAGE INTO 63 FEATURES
# =========================================================

def get_landmarks(image, detector):

    # Convert BGR → RGB
    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Convert to MediaPipe image
    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    # Detect hand
    result = detector.detect(mp_image)

    # No hand found
    if not result.hand_landmarks:
        return None

    # Take first hand
    landmarks = result.hand_landmarks[0]

    # Wrist coordinates
    wrist_x = landmarks[0].x
    wrist_y = landmarks[0].y
    wrist_z = landmarks[0].z

    features = []

    # Store x, y, z for all 21 landmarks
    for landmark in landmarks:

        x = landmark.x - wrist_x
        y = landmark.y - wrist_y
        z = landmark.z - wrist_z

        features.extend([x, y, z])

    return features


# =========================================================
# 3. CREATE MEDIAPIPE HAND DETECTOR FOR IMAGES
# =========================================================

image_options = HandLandmarkerOptions(
    base_options=BaseOptions(
        model_asset_path="hand_landmarker.task"
    ),
    running_mode=RunningMode.IMAGE,
    num_hands=1
)


# =========================================================
# 4. READ DATASET
# =========================================================

X = []
y = []

dataset_path = "dataset"

print("Reading dataset...")


with HandLandmarker.create_from_options(image_options) as detector:

    # Go through A, B, C ... Z folders
    for letter in sorted(os.listdir(dataset_path)):

        letter_path = os.path.join(dataset_path, letter)

        # Ignore files
        if not os.path.isdir(letter_path):
            continue

        print("Processing:", letter)

        # Go through images in that letter's folder
        for filename in os.listdir(letter_path):

            image_path = os.path.join(letter_path, filename)

            image = cv2.imread(image_path)

            if image is None:
                continue

            # Extract 21 landmarks
            features = get_landmarks(image, detector)

            # If a hand was detected
            if features is not None:

                X.append(features)
                y.append(letter)


print("\nDataset processing complete.")

print("Total usable images:", len(X))


# =========================================================
# 5. CONVERT TO NUMPY ARRAY
# =========================================================

X = np.array(X)
y = np.array(y)

print("Feature shape:", X.shape)


# =========================================================
# 6. SPLIT INTO TRAINING AND TESTING DATA
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining images:", len(X_train))
print("Testing images:", len(X_test))


# =========================================================
# 7. CREATE RANDOM FOREST
# =========================================================

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)


# =========================================================
# 8. TRAIN RANDOM FOREST
# =========================================================

print("\nTraining Random Forest...")

model.fit(X_train, y_train)

print("Training complete!")


# =========================================================
# 9. TEST THE MODEL
# =========================================================

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("\nTest Accuracy:", accuracy * 100, "%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


# =========================================================
# 10. SAVE THE TRAINED MODEL
# =========================================================

joblib.dump(model, "asl_random_forest.pkl")

print("\nModel saved as asl_random_forest.pkl")


# =========================================================
# 11. LIVE CAMERA DETECTION
# =========================================================

live_options = HandLandmarkerOptions(
    base_options=BaseOptions(
        model_asset_path="hand_landmarker.task"
    ),
    running_mode=RunningMode.VIDEO,
    num_hands=1
)


with HandLandmarker.create_from_options(live_options) as detector:

    cap = cv2.VideoCapture(0)

    timestamp = 0

    while True:

        success, frame = cap.read()

        if not success:
            break

        # Mirror camera
        frame = cv2.flip(frame, 1)

        # Convert BGR → RGB
        rgb = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        # MediaPipe image
        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb
        )

        # Detect hand
        result = detector.detect_for_video(
            mp_image,
            timestamp
        )

        timestamp += 1

        # If hand detected
        if result.hand_landmarks:

            landmarks = result.hand_landmarks[0]

            # Wrist
            wrist_x = landmarks[0].x
            wrist_y = landmarks[0].y
            wrist_z = landmarks[0].z

            features = []

            # Create same 63 features
            for landmark in landmarks:

                x = landmark.x - wrist_x
                y = landmark.y - wrist_y
                z = landmark.z - wrist_z

                features.extend([x, y, z])

            # Convert to numpy
            features = np.array(features).reshape(1, -1)

            # Predict letter
            prediction = model.predict(features)[0]

            # Get confidence
            probabilities = model.predict_proba(features)

            confidence = np.max(probabilities) * 100

            # Display prediction
            cv2.putText(
                frame,
                "Letter: " + prediction,
                (30, 60),
                cv2.FONT_HERSHEY_SIMPLEX,
                1.5,
                (0, 255, 0),
                3
            )

            # Display confidence
            cv2.putText(
                frame,
                f"Confidence: {confidence:.1f}%",
                (30, 100),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (255, 255, 255),
                2
            )

            # Draw landmarks
            for landmark in landmarks:

                x = int(
                    landmark.x * frame.shape[1]
                )

                y = int(
                    landmark.y * frame.shape[0]
                )

                cv2.circle(
                    frame,
                    (x, y),
                    5,
                    (0, 255, 0),
                    -1
                )

        else:

            cv2.putText(
                frame,
                "No hand detected",
                (30, 60),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 0, 255),
                2
            )

        # Show camera
        cv2.imshow(
            "ASL Gesture Translator",
            frame
        )

        # Press q to quit
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()

Reading dataset...
Processing: A
Processing: B
Processing: C
Processing: D
Processing: E
Processing: F
Processing: G
Processing: H
Processing: I
Processing: K
Processing: L
Processing: M
Processing: N
Processing: O
Processing: P
Processing: Q
Processing: R
Processing: S
Processing: T
Processing: U
Processing: V
Processing: W
Processing: X
Processing: Y

Dataset processing complete.
Total usable images: 2393
Feature shape: (2393, 63)

Training images: 1914
Testing images: 479

Training Random Forest...
Training complete!

Test Accuracy: 100.0 %

Classification Report:
              precision    recall  f1-score   support

           A       1.00      1.00      1.00        20
           B       1.00      1.00      1.00        20
           C       1.00      1.00      1.00        20
           D       1.00      1.00      1.00        21
           E       1.00      1.00      1.00        20
           F       1.00      1.00      1.00        21
           G       1.00      1.00      1.00    